# LeetCode 4097 — Minimum Value of Merged Integers bằng Python

Link đề: https://leetcode.com/problems/minimum-value-of-merged-integers/ · Độ khó: **Easy** · Chủ đề: Array, String, Math (ghép số theo cặp)

Skeleton LeetCode:

```python
class Solution:
    def minMergedInteger(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc (Weekly Contest 526, Q1); mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force viết độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ghép các phần tử **theo cặp cố định** `(nums[0], nums[1])`, `(nums[2], nums[3])`, ..., rồi lấy **giá trị nhỏ nhất** trong các số sau khi ghép; "ghép" nghĩa là **nối chữ số** (concatenate), **không phải cộng**.
2. Biết đổi qua lại giữa **số và chuỗi** (`str(x)`, `int(s)`), và vì sao `str(23) + str(55) == "2355"` còn `23 + 55 == 78`.
3. Đi từ cách **nối chuỗi rồi đổi về `int`** → **ghép bằng toán học** `a * 10^d + b` (với `d` là số chữ số của `b`) → **so sánh trực tiếp bằng (độ dài, chuỗi)** mà không đổi về số → **cắt lát bước nhảy 2** với `zip`.
4. Quen với kỹ thuật duyệt mảng **theo bước nhảy 2** (`range(0, n, 2)`) và đọc hai phần tử `i`, `i+1`.
5. Tránh các lỗi kinh điển: cộng thay vì nối, `a * 10 + b` (chỉ đúng khi `b` có một chữ số), ghép **chồng lấn** các cặp, so sánh **chuỗi theo từ điển** thay vì theo giá trị số.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài **chẵn** `n`.

Bạn phải **ghép (merge)** các phần tử theo cặp: `nums[0]` với `nums[1]`, `nums[2]` với `nums[3]`, ... , cuối cùng `nums[n-2]` với `nums[n-1]`.

Trả về giá trị **nhỏ nhất** trong các số nguyên sau khi ghép.

**Ghép** hai số nguyên nghĩa là viết **toàn bộ chữ số thập phân của số thứ nhất, rồi tới toàn bộ chữ số thập phân của số thứ hai**.

**Ví dụ 1**

- Input: `nums = [23,55,42,21]`
- Output: `2355`
- Giải thích: ghép `23` và `55` được `2355`; ghép `42` và `21` được `4221`. `min(2355, 4221) = 2355`.

**Ví dụ 2**

- Input: `nums = [34,7,11,11]`
- Output: `347`
- Giải thích: ghép `34` và `7` được `347`; ghép `11` và `11` được `1111`. `min(347, 1111) = 347`.

**Ví dụ 3**

- Input: `nums = [9,20]`
- Output: `920`
- Giải thích: chỉ có một cặp: ghép `9` và `20` được `920`, nên giá trị nhỏ nhất là `920`.

**Ràng buộc:** `2 <= n == nums.length <= 100`; `n` là số chẵn; `1 <= nums[i] <= 1000`.

In [1]:
from typing import List
import random

examples = [
    ([23, 55, 42, 21], 2355),
    ([34, 7, 11, 11], 347),
    ([9, 20], 920),
]
for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[23, 55, 42, 21] -> kỳ vọng 2355
nums=[34, 7, 11, 11] -> kỳ vọng 347
nums=[9, 20] -> kỳ vọng 920


## 3. Kiến thức nền

### 3.1. Số và chuỗi: `str()` và `int()`

`str(23)` cho chuỗi `"23"`; `int("23")` cho số `23`. Toán tử `+` với **chuỗi** là **nối chuỗi**: `"23" + "55" == "2355"`; với **số** là phép cộng: `23 + 55 == 78`. Đề yêu cầu nối chữ số, nên ta phải làm việc trên chuỗi (hoặc dùng toán học tương đương).

In [2]:
print(str(23), type(str(23)))              # 23 <class 'str'>
print(int("23"), type(int("23")))          # 23 <class 'int'>
print("23" + "55", 23 + 55)                # 2355 78
print(int("23" + "55") + 1)                # 2356: sau khi nối ta đổi về số để tính toán

23 <class 'str'>
23 <class 'int'>
2355 78
2356


### 3.2. Duyệt theo bước nhảy 2: `range(start, stop, step)`

`range(0, n, 2)` sinh `0, 2, 4, ...` — chỉ số **đầu mỗi cặp**. Với `n` chẵn, cặp thứ `k` là `(nums[2k], nums[2k+1])`. Đọc `nums[i + 1]` an toàn vì `n` chẵn nên `i + 1 <= n - 1`.

In [3]:
nums = [23, 55, 42, 21]
for i in range(0, len(nums), 2):
    print(f"cặp bắt đầu ở i={i}: ({nums[i]}, {nums[i + 1]})")

cặp bắt đầu ở i=0: (23, 55)
cặp bắt đầu ở i=2: (42, 21)


### 3.3. Lát cắt có bước: `nums[::2]` và `nums[1::2]`

`nums[::2]` lấy phần tử ở chỉ số **chẵn** (`0, 2, 4, ...`), `nums[1::2]` lấy ở chỉ số **lẻ**. `zip(nums[::2], nums[1::2])` ghép chúng thành các cặp `(nums[0], nums[1])`, `(nums[2], nums[3])`, ... đúng như đề.

In [4]:
nums = [34, 7, 11, 11]
print(nums[::2], nums[1::2])                 # [34, 11] [7, 11]
print(list(zip(nums[::2], nums[1::2])))      # [(34, 7), (11, 11)]

[34, 11] [7, 11]
[(34, 7), (11, 11)]


### 3.4. `min` và biểu thức sinh (generator)

`min(iterable)` trả phần tử nhỏ nhất. Ta có thể viết `min(f(x) for x in ...)` để tính `f` cho từng phần tử rồi lấy nhỏ nhất, **không cần tạo danh sách trung gian**.

In [5]:
print(min([5, 2, 9]))                         # 2
print(min(x * x for x in [3, -4, 2]))         # 4

2
4


### 3.5. Số chữ số của một số và lũy thừa của 10

Số `b` có `d` chữ số thì `10^(d-1) <= b < 10^d`. Ghép `a` với `b` bằng toán học: `a * 10^d + b` (dịch `a` sang trái `d` chữ số rồi cộng `b`). Ví dụ `a = 34`, `b = 7` có `d = 1`: `34 * 10 + 7 = 347`. Với `b = 20` có `d = 2`: `9 * 100 + 20 = 920`.

In [6]:
def digits(x):
    d = 0
    while x:
        d += 1
        x //= 10
    return d


for b in (7, 20, 555, 1000):
    print(b, "có", digits(b), "chữ số; len(str(b)) =", len(str(b)))
print(34 * 10 ** digits(7) + 7, 9 * 10 ** digits(20) + 20)      # 347 920

7 có 1 chữ số; len(str(b)) = 1
20 có 2 chữ số; len(str(b)) = 2
555 có 3 chữ số; len(str(b)) = 3
1000 có 4 chữ số; len(str(b)) = 4
347 920


### 3.6. So sánh số có độ dài khác nhau

Với số nguyên dương **không có chữ số 0 đứng đầu**: số có **nhiều chữ số hơn thì lớn hơn**; cùng số chữ số thì so sánh **theo từ điển** (từ trái sang phải) cũng là so sánh theo giá trị. Do đó khóa so sánh `(len(s), s)` cho chuỗi `s` sắp xếp đúng thứ tự giá trị số. Chỉ so sánh chuỗi thuần (`"2355" < "91"`) là **sai** (xem phần lỗi).

In [7]:
print("920" < "2355", "(so chuỗi theo từ điển)")       # False: vì '9' > '2'
print(920 < 2355, "(so số)")                           # True
key = lambda s: (len(s), s)
print(key("920") < key("2355"))                         # True: đúng thứ tự số

False (so chuỗi theo từ điển)
True (so số)
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [34, 7, 11, 11]`:

1. Chia thành các cặp **không giao nhau**: `(34, 7)` và `(11, 11)`.
2. Ghép từng cặp: `34` và `7` → viết `34` rồi `7` ra `347`; `11` và `11` → `1111`.
3. So sánh `347` và `1111`: `347` nhỏ hơn.

Không có thuật toán phức tạp nào ở đây: bài kiểm tra bạn có nối đúng chữ số, duyệt đúng cặp, và lấy `min` theo **giá trị số** không. Với `n <= 100` và mỗi số `<= 1000` (tối đa 4 chữ số) số ghép có tối đa `8` chữ số — vừa trong `int` thông thường.

## 5. Cách 1 — Brute Force / baseline: nối chuỗi, đổi về `int`, tìm min bằng danh sách

**Ý tưởng.** Làm đúng từng bước của đề:

1. Tạo danh sách `merged` rỗng.
2. Với `i = 0, 2, 4, ...`: nối `str(nums[i]) + str(nums[i + 1])`, đổi về `int`, thêm vào `merged`.
3. Trả `min(merged)`.

In [8]:
def merged_min_list(nums):
    merged = []
    for i in range(0, len(nums), 2):
        text = str(nums[i]) + str(nums[i + 1])     # nối chữ số
        merged.append(int(text))                    # đổi về số để so sánh đúng giá trị
    return min(merged)


for nums, expected in examples:
    print(nums, "->", merged_min_list(nums), "(kỳ vọng", expected, ")")

[23, 55, 42, 21] -> 2355 (kỳ vọng 2355 )
[34, 7, 11, 11] -> 347 (kỳ vọng 347 )
[9, 20] -> 920 (kỳ vọng 920 )


In [9]:
# Quan sát danh sách các số đã ghép của ví dụ 2
nums = [34, 7, 11, 11]
merged = [int(str(nums[i]) + str(nums[i + 1])) for i in range(0, len(nums), 2)]
print("merged =", merged, "-> min =", min(merged))

merged = [347, 1111] -> min = 347


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có `n/2` cặp, mỗi cặp nối/đổi `O(1)` (tối đa 8 chữ số): **Time `O(n)`**. Danh sách `merged` dài `n/2`: **Space `O(n)`**.

**Nút thắt.** Ta chỉ cần giá trị nhỏ nhất nên không cần giữ cả danh sách `merged`; có thể cập nhật `best` ngay khi tính xong từng cặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1:** "giữ nhỏ nhất đến giờ" thay cho danh sách: `best = min(best, giá trị cặp hiện tại)` → **Space `O(1)`**.

**Quan sát 2:** có thể **bỏ chuỗi** khi ghép: nếu `b` có `d` chữ số thì `ghép(a, b) = a * 10^d + b`. Vì `b <= 1000` nên `d` chỉ nhận `1, 2, 3, 4`.

**Quan sát 3:** có thể **bỏ cả `int`**: so sánh các chuỗi ghép bằng khóa `(len, chuỗi)`; kết quả cuối mới đổi sang `int` một lần.

Các cách này đều `O(n)` thời gian; điểm đáng học là hiểu **bản chất "nối chữ số"** theo ba góc nhìn (chuỗi, toán học, thứ tự so sánh).

## 7. Các cách tiếp theo

### Cách 2 — Duyệt bước nhảy 2, giữ `best` (không dùng danh sách)

**Giải thích.** `best = +vô cực`; với mỗi cặp ghép bằng chuỗi, cập nhật `best = min(best, value)`.

In [10]:
def merged_min_running(nums):
    best = float("inf")
    for i in range(0, len(nums), 2):
        value = int(str(nums[i]) + str(nums[i + 1]))
        if value < best:
            best = value
    return int(best)


for nums, expected in examples:
    print(nums, "->", merged_min_running(nums), "(kỳ vọng", expected, ")")

[23, 55, 42, 21] -> 2355 (kỳ vọng 2355 )
[34, 7, 11, 11] -> 347 (kỳ vọng 347 )
[9, 20] -> 920 (kỳ vọng 920 )


In [11]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def minMergedInteger(self, nums: List[int]) -> int:
        best = float("inf")
        for i in range(0, len(nums), 2):
            value = int(str(nums[i]) + str(nums[i + 1]))
            best = min(best, value)
        return int(best)


print(Solution().minMergedInteger([34, 7, 11, 11]))   # 347

347


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Ghép bằng toán học (không dùng chuỗi)

**Giải thích.** Ghép `a` và `b`: tính `d` = số chữ số của `b`, kết quả `a * 10**d + b`. Với `b <= 1000` có thể tính `d` bằng vài phép so sánh; ở đây dùng vòng `while` cho tổng quát.

In [12]:
def merge_math(a, b):
    p = 1
    t = b
    while t:
        p *= 10          # p = 10^(số chữ số của b)
        t //= 10
    return a * p + b


def merged_min_math(nums):
    return min(merge_math(nums[i], nums[i + 1]) for i in range(0, len(nums), 2))


print(merge_math(34, 7), merge_math(9, 20), merge_math(1, 1000))      # 347 920 11000
for nums, expected in examples:
    print(nums, "->", merged_min_math(nums), "(kỳ vọng", expected, ")")

347 920 11000
[23, 55, 42, 21] -> 2355 (kỳ vọng 2355 )
[34, 7, 11, 11] -> 347 (kỳ vọng 347 )
[9, 20] -> 920 (kỳ vọng 920 )


**Độ phức tạp Cách 3:** **Time `O(n)`** (mỗi cặp `O(d)` với `d <= 4`), **Space `O(1)`**.

### Cách 4 — So sánh bằng khóa `(độ dài, chuỗi)` rồi mới đổi sang số

**Giải thích.** Nối chuỗi cho mỗi cặp; chọn chuỗi nhỏ nhất theo khóa `(len(s), s)` (ngắn hơn thì nhỏ hơn; cùng độ dài thì so từ điển); cuối cùng `int(...)` đúng **một lần**.

In [13]:
def merged_min_key(nums):
    texts = [str(nums[i]) + str(nums[i + 1]) for i in range(0, len(nums), 2)]
    best = min(texts, key=lambda s: (len(s), s))
    return int(best)


for nums, expected in examples:
    print(nums, "->", merged_min_key(nums), "(kỳ vọng", expected, ")")

[23, 55, 42, 21] -> 2355 (kỳ vọng 2355 )
[34, 7, 11, 11] -> 347 (kỳ vọng 347 )
[9, 20] -> 920 (kỳ vọng 920 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (danh sách các chuỗi).

### Cách 5 — Một dòng: `zip` hai lát cắt

**Giải thích.** `zip(nums[::2], nums[1::2])` cho từng cặp; ghép bằng f-string; `min(..., key=int)` lấy nhỏ nhất theo giá trị số mà vẫn giữ chuỗi.

In [14]:
def merged_min_zip(nums):
    return min(int(f"{a}{b}") for a, b in zip(nums[::2], nums[1::2]))


for nums, expected in examples:
    print(nums, "->", merged_min_zip(nums), "(kỳ vọng", expected, ")")

[23, 55, 42, 21] -> 2355 (kỳ vọng 2355 )
[34, 7, 11, 11] -> 347 (kỳ vọng 347 )
[9, 20] -> 920 (kỳ vọng 920 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (hai lát cắt; dùng `itertools.islice` để về `O(1)`).

> Cách 2 (hoặc Cách 5 nếu thích ngắn) là bản nộp tốt. Cách 3 và 4 giúp hiểu rõ phép "nối chữ số".

## 8. TRACE từng bước

Ta in từng cặp, chuỗi ghép, giá trị và `best` sau mỗi vòng lặp của Cách 2.

In [15]:
def trace_merged_min(nums):
    print("nums =", nums)
    best = None
    for i in range(0, len(nums), 2):
        a, b = nums[i], nums[i + 1]
        text = str(a) + str(b)
        value = int(text)
        old = best
        best = value if best is None or value < best else best
        print(f"i={i}: cặp ({a}, {b}) -> {text!r} -> {value}; best trước={old}, sau={best}")
    print("kết quả:", best)
    return best


trace_merged_min([23, 55, 42, 21])
print()
trace_merged_min([23, 55, 9, 1, 100, 2])

nums = [23, 55, 42, 21]
i=0: cặp (23, 55) -> '2355' -> 2355; best trước=None, sau=2355
i=2: cặp (42, 21) -> '4221' -> 4221; best trước=2355, sau=2355
kết quả: 2355

nums = [23, 55, 9, 1, 100, 2]
i=0: cặp (23, 55) -> '2355' -> 2355; best trước=None, sau=2355
i=2: cặp (9, 1) -> '91' -> 91; best trước=2355, sau=91
i=4: cặp (100, 2) -> '1002' -> 1002; best trước=91, sau=91
kết quả: 91


91

Bảng tóm tắt (ví dụ 1, `nums = [23, 55, 42, 21]`):

| `i` | cặp | chuỗi ghép | giá trị | `best` sau |
|---|---|---|---|---|
| 0 | `(23, 55)` | `"2355"` | 2355 | 2355 |
| 2 | `(42, 21)` | `"4221"` | 4221 | 2355 |

Ví dụ thứ hai (`[23, 55, 9, 1, 100, 2]`) cho `2355`, `91`, `1002`; số nhỏ nhất là `91` dù chuỗi `"2355"` đứng trước theo từ điển — vì `91` có **ít chữ số hơn**.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (giữ `best`). Đúng `class Solution`, đúng tên method `minMergedInteger`.

In [16]:
class Solution:
    def minMergedInteger(self, nums: List[int]) -> int:
        best = float("inf")
        for i in range(0, len(nums), 2):
            value = int(str(nums[i]) + str(nums[i + 1]))
            best = min(best, value)
        return int(best)


sol = Solution()
for nums, expected in examples:
    assert sol.minMergedInteger(nums) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Cặp đúng:** `range(0, n, 2)` cho các chỉ số `0, 2, ..., n-2`; vì `n` chẵn nên `i + 1 <= n-1` luôn hợp lệ, và các cặp **không giao nhau, phủ hết** mảng — đúng đề.
- **Ghép đúng:** `str(a) + str(b)` viết toàn bộ chữ số của `a` rồi của `b` (không có số 0 thừa vì `nums[i] >= 1`), `int(...)` đổi về đúng giá trị số.
- **Min đúng:** cập nhật `best = min(best, value)` qua mọi cặp, nên cuối cùng `best` là nhỏ nhất của **tất cả** giá trị ghép (invariant: sau mỗi vòng, `best` là nhỏ nhất của các cặp đã xét).
- **Kiểu dữ liệu:** `best` khởi tạo `inf` (float) nhưng sau cặp đầu tiên luôn bằng một `int` thực (vì `int < inf`); `int(best)` ở cuối an toàn vì có ít nhất một cặp (`n >= 2`).
- **Ghép toán học (Cách 3)** tương đương ghép chuỗi vì `b` có `d` chữ số nên `a * 10^d` dành đúng `d` chữ số cuối cho `b`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa `100`), mỗi số có tối đa `4` chữ số.

- **Cách 1 (danh sách):** Time `O(n)`, Space `O(n)`.
- **Cách 2 (giữ `best`):** Time `O(n)`, Space `O(1)`.
- **Cách 3 (toán học):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (khóa `(len, s)`):** Time `O(n)`, Space `O(n)`.
- **Cách 5 (`zip`):** Time `O(n)`, Space `O(n)` do lát cắt.

Mọi cách đều **phải đọc từng phần tử ít nhất một lần**, nên `O(n)` là tối ưu về bậc. Không có sự khác biệt average/worst vì thuật toán không phụ thuộc giá trị.

## 12. Bộ test

Ta viết `oracle` độc lập: ghép bằng cách **tự tính số chữ số trong chuỗi bằng `len`** và cộng dồn từng chữ số (`value = value * 10 + digit`), không dùng `int(a+b)`, rồi so sánh với mọi cách.

In [17]:
def oracle(nums):
    """Brute force độc lập: dựng số ghép từng chữ số một."""
    n = len(nums)
    results = []
    for k in range(n // 2):
        a, b = nums[2 * k], nums[2 * k + 1]
        value = 0
        for ch in str(a):
            value = value * 10 + (ord(ch) - ord("0"))
        for ch in str(b):
            value = value * 10 + (ord(ch) - ord("0"))
        results.append(value)
    smallest = results[0]
    for v in results:
        if v < smallest:
            smallest = v
    return smallest


impls = {
    "list": merged_min_list,
    "running": merged_min_running,
    "math": merged_min_math,
    "key": merged_min_key,
    "zip": merged_min_zip,
    "submit": sol.minMergedInteger,
}

for nums, expected in examples:
    assert oracle(nums) == expected
    for key, fn in impls.items():
        assert fn(nums) == expected, (key, nums, fn(nums), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [18]:
edge_cases = [
    ([1, 1], 11, "n = 2 nhỏ nhất, hai số một chữ số"),
    ([1000, 1000], 10001000, "giá trị tối đa: 8 chữ số"),
    ([1, 1000], 11000, "một chữ số rồi bốn chữ số"),
    ([1000, 1], 10001, "bốn chữ số rồi một chữ số"),
    ([5, 5, 5, 5], 55, "mọi cặp bằng nhau"),
    ([23, 55, 9, 1], 91, "ít chữ số hơn nhưng bắt đầu bằng chữ số lớn: 91 < 2355"),
    ([10, 10, 1, 100], 1010, "10|10 = 1010; 1|100 = 1100 -> nhỏ nhất là 1010"),
    ([100, 1, 10, 10], 1001, "100|1 = 1001; 10|10 = 1010 -> nhỏ nhất là 1001"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == expected, (label, oracle(nums), expected)
    for key, fn in impls.items():
        assert fn(nums) == expected, (key, label, nums, fn(nums), expected)
    print("PASS", label)

PASS n = 2 nhỏ nhất, hai số một chữ số
PASS giá trị tối đa: 8 chữ số
PASS một chữ số rồi bốn chữ số
PASS bốn chữ số rồi một chữ số
PASS mọi cặp bằng nhau
PASS ít chữ số hơn nhưng bắt đầu bằng chữ số lớn: 91 < 2355
PASS 10|10 = 1010; 1|100 = 1100 -> nhỏ nhất là 1010
PASS 100|1 = 1001; 10|10 = 1010 -> nhỏ nhất là 1001


### 12.2. Tính chất và phản ví dụ

- Kết quả luôn là **một trong các số ghép** nên `>= 11` (hai số nhỏ nhất `1` và `1`) và `<= 10001000`.
- Hoán vị **các cặp** (đổi chỗ cặp này với cặp kia) không đổi kết quả; hoán vị **bên trong** một cặp thì có thể đổi (`"34"+"7"` khác `"7"+"34"`).
- **Phản ví dụ** cho lỗi so sánh chuỗi theo từ điển: `[23, 55, 9, 1]` cho chuỗi `"2355"` và `"91"`; `min` theo chuỗi chọn `"2355"` (vì `'2' < '9'`) nhưng giá trị nhỏ nhất là `91`.

In [19]:
random.seed(97)
for _ in range(500):
    n = 2 * random.randint(1, 8)
    nums = [random.randint(1, 1000) for _ in range(n)]
    ans = sol.minMergedInteger(nums)
    assert 11 <= ans <= 10001000
    pairs = [(nums[i], nums[i + 1]) for i in range(0, n, 2)]
    random.shuffle(pairs)
    shuffled = [x for p in pairs for x in p]
    assert sol.minMergedInteger(shuffled) == ans
print("PASS tính chất (cận, hoán vị cặp) trên 500 mảng ngẫu nhiên")


def sai_min_chuoi(nums):
    texts = [str(nums[i]) + str(nums[i + 1]) for i in range(0, len(nums), 2)]
    return int(min(texts))                      # SAI: min theo từ điển


bad = [23, 55, 9, 1]
print("phản ví dụ:", bad, "-> đúng", sol.minMergedInteger(bad), "| so chuỗi:", sai_min_chuoi(bad))
assert sai_min_chuoi(bad) != sol.minMergedInteger(bad)

PASS tính chất (cận, hoán vị cặp) trên 500 mảng ngẫu nhiên
phản ví dụ: [23, 55, 9, 1] -> đúng 91 | so chuỗi: 2355


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(4097)
for trial in range(3000):
    n = 2 * random.randint(1, 50)
    hi = random.choice([9, 99, 999, 1000])
    nums = [random.randint(1, hi) for _ in range(n)]
    truth = oracle(nums)
    for key, fn in impls.items():
        assert fn(nums) == truth, (key, nums, fn(nums), truth)
print("PASS 3000 mảng ngẫu nhiên (n tới 100, độ dài chữ số trộn lẫn): mọi cách khớp oracle")

PASS 3000 mảng ngẫu nhiên (n tới 100, độ dài chữ số trộn lẫn): mọi cách khớp oracle


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng thay vì nối chữ số
#### Code sai

In [21]:
def sai_cong(nums):
    return min(nums[i] + nums[i + 1] for i in range(0, len(nums), 2))     # SAI: cộng


print("đúng:", Solution().minMergedInteger([23, 55, 42, 21]), "| sai (cộng):", sai_cong([23, 55, 42, 21]))

đúng: 2355 | sai (cộng): 63


**Vì sao sai:** "ghép" là **viết hai số cạnh nhau** (`23` và `55` thành `2355`), không phải cộng (`78`). **Code đúng:** `int(str(a) + str(b))`.

### Lỗi 2: Dùng `a * 10 + b`
#### Code sai

In [22]:
def sai_nhan_10(nums):
    return min(nums[i] * 10 + nums[i + 1] for i in range(0, len(nums), 2))   # SAI khi b nhiều chữ số


print("đúng:", Solution().minMergedInteger([23, 55, 42, 21]), "| sai (a*10+b):", sai_nhan_10([23, 55, 42, 21]))
print("đúng:", Solution().minMergedInteger([9, 20]), "| sai (a*10+b):", sai_nhan_10([9, 20]))

đúng: 2355 | sai (a*10+b): 285
đúng: 920 | sai (a*10+b): 110


**Vì sao sai:** `a * 10 + b` chỉ dịch `a` đúng **một** chữ số, nên chỉ đúng khi `b` có **đúng một** chữ số. Với `b = 55` (hai chữ số) phải nhân `100`; với `b = 1000` phải nhân `10000`. **Code đúng:** nhân `10 ** len(str(b))`.

### Lỗi 3: Ghép chồng lấn các cặp
#### Code sai

In [23]:
def sai_chong_lan(nums):
    return min(int(str(nums[i]) + str(nums[i + 1])) for i in range(len(nums) - 1))   # SAI: bước nhảy 1


print("đúng:", Solution().minMergedInteger([90, 80, 1, 700]), "| sai (bước 1):", sai_chong_lan([90, 80, 1, 700]))

đúng: 1700 | sai (bước 1): 801


**Vì sao sai:** đề ghép các cặp **cố định** `(0,1), (2,3), ...`; bước nhảy `1` tạo thêm các cặp không có trong đề như `(nums[1], nums[2])`. Với `[90, 80, 1, 700]`: các cặp thật cho `9080` và `1700` (đáp án `1700`), nhưng cặp chồng lấn `(80, 1)` cho `801` nhỏ hơn nên bản sai trả `801`. **Code đúng:** `range(0, len(nums), 2)`.

### Lỗi 4: So sánh chuỗi theo từ điển thay vì theo giá trị
#### Code sai
Xem hàm `sai_min_chuoi` ở phần test: `min(["2355", "91"])` là `"2355"` vì ký tự `'2' < '9'`, nhưng giá trị `91 < 2355`.
#### Cách sửa
Đổi về `int` trước khi so sánh (`min(int(s) for s in texts)`), hoặc dùng khóa `(len(s), s)`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**concatenate** các chữ số" → làm việc trên **chuỗi** (`str(a) + str(b)`), rồi `int(...)` nếu cần so sánh/tính toán.
- Thấy "ghép các phần tử **theo cặp** 0-1, 2-3, ..." → `range(0, n, 2)` hoặc `zip(nums[::2], nums[1::2])`.
- Ghép bằng toán: `a * 10 ** (số chữ số của b) + b`; hay dùng khi bài cấm chuỗi hoặc cần tốc độ.
- So sánh số nguyên dương không có số 0 đứng đầu: **dài hơn thì lớn hơn**; cùng độ dài thì so từ điển.
- Ràng buộc nhỏ (`<= 100` phần tử, số `<= 1000`) → không cần lo tràn số; kết quả `<= 10001000` vừa trong `int` 32-bit.
- Luôn thử ví dụ có **độ dài chữ số khác nhau** (như `[23, 55, 9, 1]`) để bắt lỗi so sánh chuỗi.

## 15. Biến thể của bài

1. Trả **giá trị lớn nhất** thay vì nhỏ nhất.
2. Ghép theo **ba** phần tử một nhóm (`n` chia hết cho `3`).
3. **Sắp xếp** các nhóm theo giá trị ghép; hoặc trả về **chỉ số cặp** cho giá trị nhỏ nhất.
4. Cho phép **sắp xếp lại** các số để tạo số ghép nhỏ nhất (liên quan LeetCode 179 Largest Number).
5. Ghép **mọi** cặp `(i, j)` với `i != j` rồi lấy nhỏ nhất (dùng sắp xếp theo độ dài chữ số).

In [24]:
def merged_max(nums):
    return max(int(f"{a}{b}") for a, b in zip(nums[::2], nums[1::2]))


def merged_triples_min(nums):
    return min(int(f"{nums[i]}{nums[i + 1]}{nums[i + 2]}") for i in range(0, len(nums), 3))


def min_merge_any_two(nums):
    best = None
    for i in range(len(nums)):
        for j in range(len(nums)):
            if i != j:
                v = int(f"{nums[i]}{nums[j]}")
                best = v if best is None or v < best else best
    return best


print(merged_max([23, 55, 42, 21]))               # 4221
print(merged_triples_min([1, 2, 3, 4, 5, 6]))     # 123
print(min_merge_any_two([23, 55, 9, 1]))          # 19 (ghép 1 rồi 9; còn 9|1 = 91)

4221
123
19


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Danh sách | nối chuỗi, đổi `int`, `min(list)` | `O(n)` | `O(n)` | dễ hiểu nhất, nhìn thấy mọi số ghép |
| 2. Giữ `best` | cập nhật nhỏ nhất ngay trong vòng lặp | `O(n)` | `O(1)` | **cách nộp chuẩn** |
| 3. Toán học | `a * 10**d + b` | `O(n)` | `O(1)` | khi cấm chuỗi, hoặc cần tốc độ thuần số |
| 4. Khóa `(len, s)` | so sánh chuỗi bằng độ dài rồi từ điển | `O(n)` | `O(n)` | số cực dài, không muốn đổi sang `int` |
| 5. `zip` + generator | một dòng, `zip(nums[::2], nums[1::2])` | `O(n)` | `O(n)` | code ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `str(7) + str(20)`; (b) `7 + 20`; (c) `nums[::2]`, `nums[1::2]` với `nums = [3, 1, 4, 1]`; (d) `list(range(0, 6, 2))`.

**Bài 2 (tính bằng tay).** Tính đáp án của `nums = [12, 3, 5, 678, 90, 1]`: liệt kê các số ghép và chọn nhỏ nhất.

**Bài 3 (điền TODO).** Hoàn thiện `merged_min_todo(nums)`.

In [25]:
def merged_min_todo(nums):
    best = None
    # TODO 1: duyệt i = 0, 2, 4, ... bằng range(0, len(nums), 2)
    # TODO 2: nối chuỗi str(nums[i]) + str(nums[i + 1]) rồi đổi về int -> value
    # TODO 3: nếu best là None hoặc value < best thì best = value
    # TODO 4: trả best
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples:
#     assert merged_min_todo(nums) == expected

**Bài 4 (tự viết lại).** Viết `merged_min_math2(nums)` ghép **bằng toán học** (không dùng `str`), tính số chữ số bằng vòng `while`.

**Bài 5 (biến thể / mini challenge).** Viết `argmin_pair(nums)` trả về **chỉ số cặp** (0-based, tức `i // 2`) có số ghép nhỏ nhất; nếu nhiều cặp cùng nhỏ nhất thì trả cặp **đầu tiên**. Thử `[23, 55, 9, 1]` (kết quả `1`) và `[1, 1, 1, 1]` (kết quả `0`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `+` trên chuỗi là nối; `[::2]` lấy chỉ số chẵn. Bài 2: tách `[12,3]`, `[5,678]`, `[90,1]`. Bài 4: `a * 10**d + b` với `d` là số chữ số của `b`.
- **Mức 2 (vừa):** Bài 1: `'720'`, `27`, `[3, 4]` và `[1, 1]`, `[0, 2, 4]`. Bài 2: `123`, `5678`, `901` → nhỏ nhất `123`. Bài 3: `for i in range(0, len(nums), 2): value = int(str(nums[i]) + str(nums[i+1]))`. Bài 5: giữ cả `best` và `best_index`.
- **Mức 3 (sâu):** Bài 4: `p = 1; t = b; while t: p *= 10; t //= 10; value = a * p + b`. Bài 5: dùng `<` (nghiêm ngặt) để cập nhật chỉ số, nên cặp xuất hiện đầu tiên được giữ khi bằng nhau.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert str(7) + str(20) == "720" and 7 + 20 == 27
assert ([3, 1, 4, 1][::2], [3, 1, 4, 1][1::2]) == ([3, 4], [1, 1])
assert list(range(0, 6, 2)) == [0, 2, 4]

# Bài 2
assert Solution().minMergedInteger([12, 3, 5, 678, 90, 1]) == 123


# Bài 3
def merged_min_todo_ans(nums):
    best = None
    for i in range(0, len(nums), 2):
        value = int(str(nums[i]) + str(nums[i + 1]))
        if best is None or value < best:
            best = value
    return best


for nums, expected in examples:
    assert merged_min_todo_ans(nums) == expected


# Bài 4
def merged_min_math2(nums):
    best = None
    for i in range(0, len(nums), 2):
        a, b = nums[i], nums[i + 1]
        p, t = 1, b
        while t:
            p *= 10
            t //= 10
        value = a * p + b
        if best is None or value < best:
            best = value
    return best


for nums, expected in examples:
    assert merged_min_math2(nums) == expected


# Bài 5
def argmin_pair(nums):
    best, best_index = None, -1
    for i in range(0, len(nums), 2):
        value = int(str(nums[i]) + str(nums[i + 1]))
        if best is None or value < best:
            best, best_index = value, i // 2
    return best_index


assert argmin_pair([23, 55, 9, 1]) == 1 and argmin_pair([1, 1, 1, 1]) == 0
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ghép theo cặp, giữ min)

```python
class Solution:
    def minMergedInteger(self, nums: List[int]) -> int:
        best = float("inf")
        for i in range(0, len(nums), 2):
            best = min(best, int(str(nums[i]) + str(nums[i + 1])))
        return int(best)
```

**Checklist**

- [ ] "Ghép" là **nối chữ số** (`str(a) + str(b)`), không phải cộng, không phải `a * 10 + b` cố định?
- [ ] Duyệt cặp cố định với `range(0, n, 2)` (bước nhảy 2, không chồng lấn)?
- [ ] So sánh theo **giá trị số** (đã `int(...)`), không so chuỗi theo từ điển?
- [ ] Đã thử `n = 2`, cặp có độ dài chữ số khác nhau, và `1000`?
- [ ] Đã đối chiếu với brute force tự dựng số từng chữ số?
- [ ] Kết quả có kiểu `int` (không bị `float('inf')` lọt ra)?
- [ ] Skeleton đúng `class Solution` / `def minMergedInteger(self, nums: List[int]) -> int`?